# S2–S4 — Baseline, training and evaluation in Colab

This notebook only **calls the package** (`src/anthro`). All logic lives there, reviewed and tested,
so what runs here is exactly what is in the repository (see `src/anthro/cli.py`).

Before running: **Runtime → Change runtime type → T4 GPU**. Then Run all.
Data, image caches and trained models are kept on your Google Drive; nothing is saved in the repo.

## 1. Drive and data (restored from Drive after the first run, D-010)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE = "/content/drive/MyDrive/anthro"
DRIVE_ARCHIVE = "/content/drive/MyDrive/datasets/bodym.tar"
DATA_DIR = "/content/bodym"
os.makedirs(DRIVE, exist_ok=True)

if os.path.exists(DATA_DIR):
    print("Data already on this runtime.")
elif os.path.exists(DRIVE_ARCHIVE):
    !tar -xf "{DRIVE_ARCHIVE}" -C /content
else:
    !pip -q install awscli
    !aws s3 sync --no-sign-request s3://amazon-bodym/ {DATA_DIR} --only-show-errors
    os.makedirs(os.path.dirname(DRIVE_ARCHIVE), exist_ok=True)
    !tar -cf "{DRIVE_ARCHIVE}" -C /content bodym

## 2. Get the code
Set `BRANCH` to the branch being reviewed (e.g. `s3-photo-model`), or `main` once merged.
The commit printed below goes into the results table, so every number can be traced.

In [ ]:
BRANCH = "s1-s3-model-pipeline"  # set to "main" once PR #1 is merged
REPO = "https://github.com/Anthropometry/Anthropometry_Accademic_Model.git"
!rm -rf /content/repo && git clone -q --branch {BRANCH} {REPO} /content/repo
# Colab already has a CUDA build of torch; installing the package reuses it.
!pip -q install -e /content/repo
!git -C /content/repo log -1 --format="commit %h  %s"

## 3. Step 2 — height + weight baseline (per person)

In [ ]:
!anthro baseline --data {DATA_DIR} --test testA testB

## 4. Step 3 — train the photo model
About 30–60 minutes on a T4 (not yet timed). The best epoch is saved to Drive as `best.pt`;
`train_log.json` records every epoch. If Colab disconnects, rerun from step 1: the image cache on
Drive makes the restart quick.

In [ ]:
RUN = f"{DRIVE}/runs/r1"
!anthro train --data {DATA_DIR} --out {RUN} --cache {DRIVE}/cache --epochs 25

## 5. Step 4 — evaluate per person, next to the baseline
Prints, for Test-A and Test-B: baseline vs photo model (MAE, 90th percentile, % within ±1.5 cm),
consistency across each person's photos, and results by sex and BMI band.
Copy these tables into `docs/RESULTS.md` with the date, commit and data fingerprint.

In [ ]:
!anthro evaluate --data {DATA_DIR} --ckpt {RUN}/best.pt --cache {DRIVE}/cache

## Before committing this notebook
**Edit → Clear all outputs**, then save. CI refuses notebooks with saved outputs, so no data or images
reach the repository (POLICY 6).